In [1]:
# Libraries
import numpy as np
import pandas as pd
import deap
import random

from collections import defaultdict
from deap import base, creator, tools, algorithms

In [2]:
# Import prediction data
df = pd.read_csv("Results/RUL_predictions.csv")
RUL = dict(zip(df.engine_id, df.RUL_predicted))

# Set random seed

random.seed(33)

In [3]:
# Various helper functions and other structures
def get_engine_due_date(engine_id, RUL):
    '''Computes the engine's safety due date (ts = t1 + Rj - 1), t1 = 1'''
    t1 = 1
    return t1 + RUL[engine_id] + 1

def get_maintenance_duration(engine_id, team_type):
    '''Returns the days to perform maintenance for the input team type and engine ID'''
    # First get the days for team A
    mu_A = None
    if 1 <= engine_id <= 20:
        mu_A = 5
    elif 21 <= engine_id <= 55:
        mu_A = 3
    elif 56 <= engine_id <= 80:
        mu_A = 4
    else:
        mu_A = 5

    # If team type A, return mu_A. Otherwise, use mu_A to calculate team B's days.
    if team_type == "A":
        return mu_A
    elif team_type == "B":
        if 1 <= engine_id <= 25:
            return mu_A - 1
        elif 26 <= engine_id <= 70:
            return mu_A + 3
        else:
            return mu_A + 2
    else:
        raise ValueError("get_maintenance_duration() received invalid team type!")
     
def get_penalty_cost(engine_id, completion_date, due_date):
    '''Computes the penalty cost given the completion and due dates of an engine'''
    # In case completed in time, no penalty
    if completion_date <= due_date:
        return 0
    
    # Get the cj value for the engine's id range
    cj = None
    if 1 <= engine_id <= 25:
        cj = 4
    elif 26 <= engine_id <= 45:
        cj = 2
    elif 46 <= engine_id <= 75:
        cj = 5
    else:
        cj = 6
    
    # Compute penalty value
    delay = completion_date - due_date
    penalty = cj * (delay ** 2)
    return min(penalty, 250) # Return if less than 250, otherwise return 250

# Team type mapping
TEAM_TYPES = {
    1: "A",
    2: "B",
    3: "A",
    4: "B"
}

In [ ]:
### MAIN GA FUNCTIONS ###

# Encoding the GA individual (chromosomes)
# General idea: [(engine, team, start_day), (engine, team, start_day), ...]
# So one gene is one assignment of maintenance; this structure should be flexible enough for all constraints and operations

# Build schedules for teams
def build_schedule(individual):
    team_jobs = defaultdict(list)

    for engine, team, start in individual:
        team_jobs[team].append((engine, start))

    return team_jobs

def is_feasible(team_jobs):
    '''Returns whether or not a provided (individual's) schedule is feasible given the constraints'''
    for team, jobs in team_jobs.items():
        jobs = sorted(jobs, key=lambda x: x[1])
        current_time = 0

        for engine, start in jobs:
            duration = get_maintenance_duration(engine, TEAM_TYPES[team])
            if start < current_time: # If the start time of a job is before the current time, the schedule is obviously not feasible.
                return False
            if start + duration - 1 > 30: # Planning horizon of 30, so job duration cannot be longer than this.
                return False
            current_time = start + duration

    return True # If no issues caught during the loops, the schedule is valid.

def evaluate(individual, RUL):
    '''
    Calculates the total penalty for a given individual.
    Trailing commas in outputs are there for DEAP format.
    '''
    team_jobs = build_schedule(individual)

    if not is_feasible(team_jobs):
        return (1e9,) # Arbitrarily large penalty for invalid individuals
    
    maintained_engines = set()
    total_penalty = 0

    # Calculate penalties for maintained engines
    for team, jobs in team_jobs.items():
        for engine, start in jobs:
            duration = get_maintenance_duration(engine, TEAM_TYPES[team])
            completion = start + duration - 1
            due_date = get_engine_due_date(engine, RUL)
            total_penalty += get_penalty_cost(engine, completion, due_date)
            maintained_engines.add(engine)

    # Calculate penalties for non-maintained engines
    for engine in range(1, 101):
        if engine not in maintained_engines:
            due = get_engine_due_date(engine, RUL)
            total_penalty += get_penalty_cost(engine, 30, due)

    return (total_penalty,)

def custom_mutation(individual, prob=0.3):
    '''
    Combines multiple small mutations into one to ensure all parts of the chromosome can be mutated.
    This includes start time, team type, engine id, and chromosome length (amount of engines maintained).
    '''
    # Re-set seed (to ensure this is reproducible)
    random.seed(33)

    # Start time mutation
    for i in range(len(individual)):
        if random.random() < prob:
            engine, team, start = individual[i]
            # Shift the start time by a random amount within reasonable range
            shift = random.randint(-3, 3)
            new_start = max(1, min(30, start + shift))
            individual[i] = (engine, team, new_start)

    # Team type mutation
    for i in range(len(individual)):
        if random.random() < prob:
            engine, team, start = individual[i]
            # Select a new team from the available 4 teams
            new_team = random.randint(1, 4)
            individual[i] = (engine, new_team, start)

    # Engine id mutation
    for i in range(len(individual)):
        if random.random() < prob:
            engine, team, start = individual[i]
            # Select a new team from the 100 available engines
            # Has a small (1/100) chance to select the same engine; should be fine given the small chance though.
            new_engine = random.randint(1, 100)
            individual[i] = (new_engine, team, start)

    # Chromosome length (# of engines maintained) mutation
    # Chance to remove engine
    if random.random() < 0.3 and len(individual) > 1: # Ensure we do not end up with chromosomes of 0 length (no engines maintained)
        individual.pop(random.randrange(len(individual)))

    # Chance to add engine
    elif random.random() > 0.7:
        individual.append((
            random.randint(1, 100),
            random.randint(1, 4),
            random.randint(1, 30)
        ))

    # Return mutated individual
    return (individual,)

def custom_crossover(parent_1, parent_2):
    '''
    Custom crossover function with the goal of keeping schedules reasonably intact.
    To that end, children inherit subsets (~half) of parent schedules, rather than single point crossover or similar.
    '''
    child_1, child_2 = [], []

    split_1 = set(random.sample(range(len(parent_1)), k=len(parent_1)//2))
    split_2 = set(random.sample(range(len(parent_2)), k=len(parent_2)//2))

    # Child 1 creation
    child_1.extend(parent_1[i] for i in split_1)
    # Make sure we do not duplicate engine maintenance
    maintained_engines_1 = {g[0] for g in child_1}
    for gene in parent_2:
        if gene[0] not in maintained_engines_1:
            child_1.append(gene)

    # Child 2 creation
    child_2.extend(parent_2[i] for i in split_2)
    # Make sure we do not duplicate engine maintenance
    maintained_engines_2 = {g[0] for g in child_2}
    for gene in parent_1:
        if gene[0] not in maintained_engines_2:
            child_2.append(gene)

    # Overwrite parents with children
    parent_1[:] = child_1
    parent_2[:] = child_2

    return parent_1, parent_2

In [5]:
# DEAP Setup
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)
model = base.Toolbox()

# Generate individuals
def create_individual():
    n_jobs = random.randint(10, 40) # Ensure varied but reasonable schedule lengths
    indiv = []
    for i in range(n_jobs):
        engine = random.randint(1, 100)
        team = random.randint(1, 4)
        start = random.randint(1, 30)
        indiv.append((engine, team, start))

    return creator.Individual(indiv)

model.register("individual", create_individual)
model.register("population", tools.initRepeat, list, model.individual)

# Placeholder operators (TODO: better functions and run the model)
model.register("evaluate", evaluate, RUL=RUL)
model.register("mate", custom_crossover)
model.register("mutate", custom_mutation, prob=0.3)
model.register("select", tools.selTournament, tournsize=3)

In [6]:
# GA Running code
def run_ga(RUL, population_size=80, generations=60, cxpb=0.8, mutpb=0.4): #TODO: justify and/or change default numbers
    '''
    Runs the genetic algorithm using the functions defined previously.
    '''
    # Intialize population
    population = model.population(n=population_size)
    # Keep track of best solution
    track_best_sol = tools.HallOfFame(1)
    # Track useful statistics
    stats = tools.Statistics(lambda indiv: indiv.fitness_values[0])
    stats.register("avg", lambda x: sum(x)/len(x))
    stats.register("min", min)
    stats.register("max", max)

    # Setup algorithm
    population, log = algorithms.eaSimple(population, model, cxpb=cxpb, mutpb=mutpb, ngen=generations, stats=stats, halloffame=track_best_sol, verbose=True)

    # Save final best solution
    best = track_best_sol[0]

    return best, log

# Run GA
best_solution, log = run_ga(RUL)
print(best_solution, log)

AttributeError: 'Individual' object has no attribute 'fitness_values'